# Notebook 12.2 — RLCD for Reasoning: Teaching a Small Model to Reason *Without* a Verifier

### The question this notebook asks

Notebook 12 taught `Qwen2.5-0.5B-Instruct` to reason with GRPO. Its reward came
from a **verifier**: a program that checks whether the final answer is right.
Many real tasks have no such program (helpfulness, writing quality, "is this
explanation careful?"). Can we still bring reasoning to the model?

**RLCD** (Reinforcement Learning from Contrastive Distillation, notebook 11)
says yes. Prompt the model twice with *opposite principles* and treat the
difference between the two responses as a free preference label:

| | system prompt | expected response |
|---|---|---|
| $p_+$ | "be careful: reason step by step in `<think>`, double-check, then `<answer>`" | careful, structured, usually right |
| $p_-$ | "answer immediately with a quick guess, no working, no tags" | careless, unstructured, often wrong |

Each pair is stored as **(plain prompt, chosen $= o_+$, rejected $= o_-$)**.
We never ask anyone (or anything) which answer is correct. The label is
**correct by construction**, as long as the model actually follows the two
principles — we'll measure that.

### What we build

| step | `trl` | hand-written PyTorch |
|---|---|---|
| 0. contrast diagnostic | — | verifier used **only to measure** label quality |
| 1–2. contrastive pairs | — | generation with $p_+$ / $p_-$ |
| 3. reward model (Bradley–Terry) | `RewardTrainer` | `nn.Module` + training loop |
| 4a. RL with the RLCD reward model | `GRPOTrainer` (reward = RM score) | *(see notebook 12 / 12.3 for hand-written RL)* |
| 4b. skip the RM: preference optimization | `DPOTrainer` | DPO loss + training loop |

All policies are evaluated with **exactly notebook 12's protocol**: same 48
held-out problems, same system prompt, greedy decoding. Results go to
`results/12_2_rlcd.json` for the final comparison in **notebook 12.3**.

> **The verifier is never used as a training signal in this notebook.** It
> appears only in evaluation and diagnostics, as a measuring stick. That's the
> whole point: RLCD must succeed without it.

> **Hardware.** Written for a CUDA GPU (bf16 when supported). On an A100,
> expect very roughly an hour in total. Lower `RL_STEPS` / epochs on smaller
> GPUs.

## 1. Setup

In [ ]:
# ---- Environment setup (uncomment what you need, then Runtime -> Restart session) ----
# !pip install -q "transformers>=4.56" "peft>=0.17" datasets accelerate "trl>=0.23"
#
# Colab-style images ship an old `torchao`; recent `peft` refuses to create LoRA layers when an incompatible
# torchao (< 0.16) is installed. Either remove it or upgrade it:
# !pip uninstall -y torchao

In [ ]:
import os, re, time, random, copy, json, hashlib, gc, warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import transformers, trl, peft
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, AutoModel
from peft import LoraConfig, get_peft_model

from trl import GRPOConfig, GRPOTrainer, DPOConfig, DPOTrainer, RewardConfig, RewardTrainer


# ---- pick the fastest available device: CUDA GPU > Apple-Silicon GPU > CPU ----
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()      # bf16: half the memory, much faster on modern GPUs
DTYPE = torch.bfloat16 if USE_BF16 else torch.float32

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed(0)

def free_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(f"device        : {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
print(f"dtype         : {DTYPE}")
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"trl           : {trl.__version__}")
print(f"peft          : {peft.__version__}")

## 2. Task, verifier and evaluation — identical to notebook 12

The same generated word problems (same seeds, so the **same** 48 held-out test
problems), the same strict verifier, and the same greedy evaluation. The
printed **test fingerprint** must match in every notebook: notebook 12.3 checks
it before comparing results.

In [ ]:
# =====================  identical to notebook 12  =====================
NAMES = ["Sam", "Mia", "Leo", "Ava", "Noah", "Zoe", "Eli", "Ivy"]
ITEMS = ["apples", "marbles", "stickers", "books", "cookies", "pencils"]

def make_problem(rng):
    """Return (question, answer_as_string). Three templates, each needing 2-3 arithmetic steps."""
    name, item = rng.choice(NAMES), rng.choice(ITEMS)
    a, b, c, d = rng.randint(2, 20), rng.randint(2, 6), rng.randint(2, 9), rng.randint(1, 10)
    kind = rng.randint(0, 2)
    if kind == 0:
        q = (f"{name} has {a} {item}. {name} buys {b} packs with {c} {item} in each pack, "
             f"then gives away {d}. How many {item} does {name} have now?")
        ans = a + b * c - d
    elif kind == 1:
        q = (f"A box holds {c} {item}. {name} fills {b} boxes and has {a} loose {item}. "
             f"{name} then loses {d}. How many {item} are left?")
        ans = b * c + a - d
    else:
        q = (f"{name} reads {c} pages a day for {b} days, then reads {a} more pages. "
             f"How many pages did {name} read in total?")
        ans = b * c + a
    return q, str(ans)

def make_dataset(n, seed):
    rng = random.Random(seed)
    return [make_problem(rng) for _ in range(n)]

TRAIN = make_dataset(400, seed=1)
train_questions = {q for q, _ in TRAIN}
TEST = [p for p in make_dataset(60, seed=123) if p[0] not in train_questions][:48]
ANSWER_OF = {q: a for q, a in TRAIN + TEST}

# A fingerprint of the test set: notebook 12.3 refuses to compare results computed on different test sets.
TEST_FINGERPRINT = hashlib.sha1("\n".join(f"{q}|{a}" for q, a in TEST).encode()).hexdigest()[:12]
print(f"train problems: {len(TRAIN)} | test problems: {len(TEST)} | overlap: {len(train_questions & {q for q, _ in TEST})}"
      f" | test fingerprint: {TEST_FINGERPRINT}")

SYSTEM_PROMPT = (
    "You are a helpful assistant. First think through the problem step by step inside "
    "<think> </think> tags, then give only the final number inside <answer> </answer> tags."
)
ANSWER_RE = re.compile(r"<answer>\s*(-?\d+)\s*</answer>")
FORMAT_RE = re.compile(r"<think>.+?</think>\s*<answer>\s*-?\d+\s*</answer>", re.S)

def parse_answer(text):
    """Strict verifier: the number inside <answer>...</answer>, else None."""
    m = ANSWER_RE.search(text)
    return m.group(1) if m else None

def follows_format(text):
    return bool(FORMAT_RE.search(text))

def format_score(text):
    r = 0.0
    if re.search(r"<think>.+?</think>", text, re.S):
        r += 0.10                                   # a reasoning block (worth little on its own)
    if ANSWER_RE.search(text):
        r += 0.25                                   # commits to a parseable answer
    if follows_format(text) and text.strip().endswith("</answer>"):
        r += 0.40                                   # the full structure: think -> answer -> stop
    return r

def correctness_score(text, answer):
    return 1.0 if parse_answer(text) == answer else 0.0

def verifier_reward(text, answer):
    """The notebook-12 reward: format (max 0.75) + correctness (1.0)."""
    return format_score(text) + correctness_score(text, answer)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"          # left-pad prompts so generation continues on the right
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def chat_prompt(question, system=SYSTEM_PROMPT):
    """System prompt + user question, rendered with the model's own chat template (ready for generation)."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_prompt(question):
    return chat_prompt(question, SYSTEM_PROMPT)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE).eval()
print(f"loaded {MODEL_NAME}: {sum(p.numel() for p in base_model.parameters()) / 1e6:.0f}M parameters on {DEVICE}")

@torch.no_grad()
def generate_texts(model, questions, system=SYSTEM_PROMPT, sample=True, temperature=0.8,
                   max_new_tokens=200, batch_size=32, seed=None):
    """Generate one completion per question (sampled or greedy)."""
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    outs = []
    for i in range(0, len(questions), batch_size):
        enc = tokenizer([chat_prompt(q, system) for q in questions[i:i + batch_size]],
                        return_tensors="pt", padding=True).to(model.device)
        kw = dict(do_sample=True, temperature=temperature, top_k=0, top_p=1.0) if sample else dict(do_sample=False)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id, **kw)
        outs += tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

@torch.no_grad()
def evaluate(model, label, problems=TEST, max_new_tokens=200, batch_size=16, show=2):
    """Identical protocol to notebook 12: held-out problems, same system prompt, GREEDY decoding."""
    t0 = time.time()
    responses = generate_texts(model, [q for q, _ in problems], sample=False,
                               max_new_tokens=max_new_tokens, batch_size=batch_size)
    answers = [a for _, a in problems]
    metrics = {
        "accuracy (strict)": float(np.mean([parse_answer(r) == a for r, a in zip(responses, answers)])),
        "format compliance": float(np.mean([follows_format(r) for r in responses])),
        "accuracy (lenient, last number)": float(np.mean([re.findall(r"-?\d+", r)[-1:] == [a] for r, a in zip(responses, answers)])),
        "mean length (tokens)": float(np.mean([len(tokenizer(r)["input_ids"]) for r in responses])),
    }
    print(f"===== {label}: evaluated {len(problems)} held-out problems in {time.time() - t0:.0f}s =====")
    for k, v in metrics.items():
        print(f"   {k:34}: {v:.3f}" if "length" not in k else f"   {k:34}: {v:.1f}")
    for j in range(show):
        print(f"\n   --- example {j + 1}: {problems[j][0]}\n   --- true answer: {problems[j][1]}\n{responses[j]}")
    return metrics, responses

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)

def method_record(metrics, responses, *, algorithm, implementation, reward_signal, train_minutes,
                  trainable_params, samples_generated, models_in_memory, notes=""):
    return {"metrics": {k: float(v) for k, v in metrics.items()}, "responses": list(responses),
            "algorithm": algorithm, "implementation": implementation, "reward_signal": reward_signal,
            "train_minutes": None if train_minutes is None else float(train_minutes),
            "trainable_params": None if trainable_params is None else int(trainable_params),
            "samples_generated": None if samples_generated is None else int(samples_generated),
            "models_in_memory": models_in_memory, "notes": notes}

def save_results(filename, methods):
    path = RESULTS_DIR / filename
    payload = {"notebook": filename, "model": MODEL_NAME, "test_fingerprint": TEST_FINGERPRINT,
               "test": [list(p) for p in TEST], "methods": methods}
    path.write_text(json.dumps(payload, indent=1))
    print(f"saved {len(methods)} method(s) to {path.resolve()}")

def selective_log_softmax(logits, targets):
    """log p(target) = z[target] - logsumexp(z), row by row: never builds a (B, T, 151936) log-prob tensor."""
    return torch.stack([row.gather(-1, tgt[:, None]).squeeze(-1).float() - torch.logsumexp(row.float(), dim=-1)
                        for row, tgt in zip(logits, targets)])

def lora_cfg(task_type="CAUSAL_LM"):
    return LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type=task_type)

def lora_params_to_fp32(model):
    """Keep the (small) trainable LoRA weights in float32: Adam updates on bf16 weights lose precision."""
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()


before_metrics, before_responses = evaluate(base_model, "BEFORE training (base model)")

## 3. Step 0 — Does the model follow the two principles? (contrast diagnostic)

RLCD's labels are only "correct by construction" if $p_+$ and $p_-$ really
**change the model's behavior** in the direction we care about. Notebook 11,
Part B showed what happens when they don't: a tiny model ignored a "be rude"
instruction, the pairs carried no signal, and nothing was learned.

So before generating a dataset, we measure the contrast on 64 training
questions. The verifier is used here **only as a measuring stick**:

- **accuracy** and **format compliance** of $o_+$ vs $o_-$;
- **label quality**: how often the "chosen" answer is *actually* at least as
  correct as the "rejected" one;
- **length**, to watch for the classic spurious cue (the reward model learning
  "longer = better" instead of "more careful = better").

In [ ]:
# The two principles. Both are complete system prompts; they differ only in the attitude they ask for.
P_POS = SYSTEM_PROMPT + (" Be careful and thorough: write out every arithmetic step inside <think> </think>, "
                         "double-check each step, and only then give the final number inside <answer> </answer>.")
P_NEG = ("You are a careless assistant. Answer immediately with a quick guess. Do not show any working, "
         "do not check anything, and do not use any tags. Reply in one short sentence.")
print("p+ :", P_POS, "\n\np- :", P_NEG)

def contrast_report(questions, answers, o_pos, o_neg, label):
    cor_pos = np.array([correctness_score(o, a) for o, a in zip(o_pos, answers)])
    cor_neg = np.array([correctness_score(o, a) for o, a in zip(o_neg, answers)])
    len_pos = np.array([len(tokenizer(o)["input_ids"]) for o in o_pos])
    len_neg = np.array([len(tokenizer(o)["input_ids"]) for o in o_neg])
    print(f"\n===== contrast diagnostic: {label} ({len(questions)} pairs) =====")
    print(f"{'':26} | {'o+ (careful)':>13} | {'o- (careless)':>13}")
    print(f"{'strict accuracy':26} | {cor_pos.mean():13.3f} | {cor_neg.mean():13.3f}")
    print(f"{'format compliance':26} | {np.mean([follows_format(o) for o in o_pos]):13.3f} | {np.mean([follows_format(o) for o in o_neg]):13.3f}")
    print(f"{'mean length (tokens)':26} | {len_pos.mean():13.1f} | {len_neg.mean():13.1f}")
    print(f"\nlabel quality, judged by the verifier (which RLCD itself never sees):")
    print(f"   chosen strictly better (o+ right, o- wrong) : {np.mean(cor_pos > cor_neg):.3f}")
    print(f"   tie (both right or both wrong)              : {np.mean(cor_pos == cor_neg):.3f}")
    print(f"   label WRONG (o- right, o+ wrong)            : {np.mean(cor_pos < cor_neg):.3f}")
    return {"acc_pos": cor_pos.mean(), "acc_neg": cor_neg.mean(), "len_pos": len_pos.mean(), "len_neg": len_neg.mean()}

diag_q = [q for q, _ in TRAIN[:64]]; diag_a = [a for _, a in TRAIN[:64]]
diag_pos = generate_texts(base_model, diag_q, system=P_POS, seed=0)
diag_neg = generate_texts(base_model, diag_q, system=P_NEG, seed=0)
_ = contrast_report(diag_q, diag_a, diag_pos, diag_neg, "64 training questions")
for i in range(2):
    print(f"\nQ: {diag_q[i]}  (answer {diag_a[i]})\n--- o+ ---\n{diag_pos[i][:400]}\n--- o- ---\n{diag_neg[i][:200]}")

**How to read the diagnostic.**

- What matters most is a **large gap in format compliance and accuracy**
  between $o_+$ and $o_-$, and a **small "label WRONG" rate**. That means the
  free labels carry real signal about careful, correct reasoning.
- **Ties** are common: both answers right, or both wrong. In a tie the pair
  still teaches *style* (structured vs. unstructured), but not *correctness*.
  RLCD labels are about the principle, not about the answer.
- A big **length gap** is expected here (careless answers are short). It's a
  warning, though: the reward model may learn length as a shortcut. Section 5
  probes for exactly that.

If the contrast is weak (similar accuracy and format on both sides), stop here
and fix the principles or use a larger model. Everything downstream would just
learn noise.

## 4. Steps 1–2 — Generate the contrastive pair dataset

One pair per training question: $o_+$ under $p_+$, $o_-$ under $p_-$, same
sampling settings. The stored prompt is the **plain** prompt (notebook 12's
`SYSTEM_PROMPT` + question). At training and evaluation time the model never
sees either principle. Learning to behave like $p_+$ *without being told to*
is the "distillation" in RLCD.

We also build pairs for the 48 **test** questions. They're used only to check
whether the reward model generalizes, never for training.

In [ ]:
def make_pairs(problems, seed):
    qs = [q for q, _ in problems]
    o_pos = generate_texts(base_model, qs, system=P_POS, seed=seed)
    o_neg = generate_texts(base_model, qs, system=P_NEG, seed=seed + 1)
    rows = [{"prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}],  # PLAIN prompt
             "chosen": [{"role": "assistant", "content": p}],                                           # o+  (label by construction)
             "rejected": [{"role": "assistant", "content": n}],                                          # o-
             "answer": a}                                                                                # kept ONLY for diagnostics
            for (q, a), p, n in zip(problems, o_pos, o_neg)]
    return rows, o_pos, o_neg

t0 = time.time()
train_pairs, tr_pos, tr_neg = make_pairs(TRAIN, seed=10)
test_pairs, te_pos, te_neg = make_pairs(TEST, seed=20)
print(f"generated {len(train_pairs)} training pairs and {len(test_pairs)} held-out pairs in {time.time() - t0:.0f}s")
train_stats = contrast_report([q for q, _ in TRAIN], [a for _, a in TRAIN], tr_pos, tr_neg, "full training set")

pairs_ds = Dataset.from_list([{k: r[k] for k in ("prompt", "chosen", "rejected")} for r in train_pairs])
print("\none training example (what the reward model / DPO will see):")
print(json.dumps({k: train_pairs[0][k] for k in ("prompt", "chosen", "rejected")}, indent=1)[:1200])

## 5. Step 3 — The reward model (Bradley–Terry), two ways

The reward model $r_\phi(x, y)$ is trained so that chosen responses score higher
than rejected ones:

$$
\mathcal{L}_{\text{RM}}(\phi) = - \mathbb{E}_{(x, y_+, y_-)} \left[ \log \sigma \left( r_\phi(x, y_+) - r_\phi(x, y_-) \right) \right] .
$$

It's the same model as notebook 7 (Bradley–Terry), here a copy of Qwen with a
**scalar head** instead of the vocabulary head, adapted with LoRA.

### Two probes that tell you whether the reward model is any good

1. **Held-out pair accuracy** — on the 48 test-question pairs: does it rank
   $o_+$ above $o_-$ for prompts it never saw?
2. **Correctness probe** — the question that matters for reasoning. We sample 4
   *plain-prompt* answers per test question from the base model, check each
   with the verifier, and ask: **does the reward model score correct answers
   higher than incorrect ones?** We report the AUC, the probability that a
   random correct answer outscores a random incorrect one (0.5 = no idea,
   1.0 = perfect). A reward model trained only on "careful vs careless" style
   might still score *careful but wrong* answers highly. That's the most
   important limitation to watch for.

In [ ]:
@torch.no_grad()
def rm_scores_generic(score_fn, questions, responses, batch_size=16):
    """Score (question, response) pairs; score_fn maps a tokenized batch to (B,) scores."""
    texts = [tokenizer.apply_chat_template([{"role": "system", "content": SYSTEM_PROMPT},
                                            {"role": "user", "content": q},
                                            {"role": "assistant", "content": r}], tokenize=False)
             for q, r in zip(questions, responses)]
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], return_tensors="pt", padding=True).to(DEVICE)
        out.append(score_fn(enc).float().cpu())
    return torch.cat(out)

# Probe data: 4 sampled plain-prompt answers per test question from the BASE model, labelled by the verifier.
probe_q = [q for q, _ in TEST for _ in range(4)]
probe_a = [a for _, a in TEST for _ in range(4)]
probe_r = generate_texts(base_model, probe_q, seed=30)
probe_correct = np.array([correctness_score(r, a) for r, a in zip(probe_r, probe_a)])
print(f"probe set: {len(probe_r)} base-model answers, {int(probe_correct.sum())} correct by the verifier")

def auc(scores_pos, scores_neg):
    """P(random positive outscores random negative); ties count half."""
    if len(scores_pos) == 0 or len(scores_neg) == 0:
        return float("nan")
    diff = scores_pos[:, None] - scores_neg[None, :]
    return float(((diff > 0).sum() + 0.5 * (diff == 0).sum()) / diff.size)

def probe_reward_model(score_fn, label):
    s_pos = rm_scores_generic(score_fn, [q for q, _ in TEST], te_pos).numpy()
    s_neg = rm_scores_generic(score_fn, [q for q, _ in TEST], te_neg).numpy()
    s_probe = rm_scores_generic(score_fn, probe_q, probe_r).numpy()
    lens = np.array([len(tokenizer(r)["input_ids"]) for r in probe_r])
    res = {"heldout_pair_acc": float(np.mean(s_pos > s_neg)),
           "correctness_auc": auc(s_probe[probe_correct == 1], s_probe[probe_correct == 0]),
           "corr_with_length": float(np.corrcoef(s_probe, lens)[0, 1])}
    print(f"===== reward-model probes: {label} =====")
    print(f"   held-out pair accuracy (o+ > o-)            : {res['heldout_pair_acc']:.3f}")
    print(f"   correctness AUC on base-model answers       : {res['correctness_auc']:.3f}   (0.5 = blind to correctness)")
    print(f"   correlation of score with response length   : {res['corr_with_length']:+.3f}   (large = length shortcut)")
    return res

### 5a. Reward model with `trl.RewardTrainer`

`AutoModelForSequenceClassification` with `num_labels=1` puts a scalar `score`
head on top of the transformer. With a `SEQ_CLS` LoRA config, `peft` trains the
adapters **and** the new head (`modules_to_save`). `RewardTrainer` applies the
chat template, batches chosen and rejected responses, and optimizes the
Bradley–Terry loss above.

In [ ]:
RM_EPOCHS = 1
rm_trl = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1, dtype=DTYPE)
rm_trl.config.pad_token_id = tokenizer.pad_token_id

rm_args = RewardConfig(
    output_dir="/tmp/nb12_2_rm_trl",
    num_train_epochs=RM_EPOCHS,
    per_device_train_batch_size=8,      # 8 pairs = 16 sequences per step
    learning_rate=1e-4,                 # LoRA tolerates (and needs) a larger LR than full fine-tuning
    max_length=512,
    logging_steps=5, report_to="none", save_strategy="no", seed=0,
    bf16=USE_BF16, use_cpu=(DEVICE == "cpu"),
)
t0 = time.time()
rm_trainer = RewardTrainer(model=rm_trl, args=rm_args, train_dataset=pairs_ds,
                           processing_class=tokenizer, peft_config=lora_cfg("SEQ_CLS"))
print(f"trainable reward-model parameters: {count_trainable(rm_trainer.model) / 1e6:.2f}M")
rm_trainer.train()
rm_trl_minutes = (time.time() - t0) / 60
print(f"\nreward model (trl) trained in {rm_trl_minutes:.1f} min")
for row in rm_trainer.state.log_history:
    if "loss" in row:
        print({k: round(float(row[k]), 3) for k in ["step", "loss", "accuracy", "margin"] if k in row})

rm_model = rm_trainer.model.eval()
rm_trl_score = lambda enc: rm_model(**enc).logits[:, 0]
rm_trl_probe = probe_reward_model(rm_trl_score, "trl RewardTrainer")

### 5b. The same reward model by hand: `nn.Module` + training loop

Nothing hidden this time:

- **Backbone**: the transformer *without* its vocabulary head (`AutoModel`),
  with LoRA adapters.
- **Pooling**: the hidden state of the **last token**, which has attended to the
  whole prompt and response. With left padding, the last real token is always
  at position $T-1$.
- **Head**: one linear layer, hidden size 896 → 1 scalar.
- **Loss**: $-\log\sigma(r_+ - r_-)$, i.e. `-F.logsigmoid(r_chosen - r_rejected)`.

Chosen and rejected responses go through the network **in one batch** (concatenated),
then are split again for the loss.

In [ ]:
class BradleyTerryRewardModel(nn.Module):
    """r_phi(x, y): LoRA-adapted transformer backbone -> last-token hidden state -> linear head -> scalar."""

    def __init__(self, model_name):
        super().__init__()
        backbone = AutoModel.from_pretrained(model_name, dtype=DTYPE)          # no LM head
        self.backbone = get_peft_model(backbone, lora_cfg("FEATURE_EXTRACTION"))
        lora_params_to_fp32(self.backbone)
        self.head = nn.Linear(backbone.config.hidden_size, 1)                   # float32 scalar head

    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state   # (B, T, H)
        pooled = h[:, -1].float()                                               # left padding -> last real token
        return self.head(pooled).squeeze(-1)                                    # (B,)

def pair_texts(rows):
    full = lambda r, key: tokenizer.apply_chat_template(r["prompt"] + r[key], tokenize=False)
    return [full(r, "chosen") for r in rows], [full(r, "rejected") for r in rows]

set_seed(0)
rm_torch = BradleyTerryRewardModel(MODEL_NAME).to(DEVICE)
rm_opt = torch.optim.AdamW([p for p in rm_torch.parameters() if p.requires_grad], lr=1e-4)
print(f"trainable parameters (LoRA + head): {count_trainable(rm_torch) / 1e6:.2f}M")

BATCH = 8
t0 = time.time(); step = 0
for epoch in range(RM_EPOCHS):
    order = list(range(len(train_pairs))); random.Random(epoch).shuffle(order)
    for i in range(0, len(order), BATCH):
        rows = [train_pairs[j] for j in order[i:i + BATCH]]
        chosen, rejected = pair_texts(rows)
        enc = tokenizer(chosen + rejected, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
        rm_torch.train()
        scores = rm_torch(enc["input_ids"], enc["attention_mask"])
        r_chosen, r_rejected = scores[:len(rows)], scores[len(rows):]
        loss = -F.logsigmoid(r_chosen - r_rejected).mean()                   # Bradley-Terry
        rm_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(rm_torch.parameters(), 1.0); rm_opt.step()
        if step % 5 == 0:
            print(f"step {step:3d} | loss {loss.item():.4f} | pair accuracy {(r_chosen > r_rejected).float().mean():.2f} "
                  f"| margin {(r_chosen - r_rejected).mean().item():+.3f}")
        step += 1
rm_torch_minutes = (time.time() - t0) / 60
print(f"\nreward model (PyTorch) trained in {rm_torch_minutes:.1f} min")

rm_torch.eval()
rm_torch_probe = probe_reward_model(lambda enc: rm_torch(enc["input_ids"], enc["attention_mask"]), "hand-written PyTorch")

Compare the two probe blocks. Both reward models should rank held-out pairs
well, since that's what they were trained on. The **correctness AUC** is the
real test: it tells you how much of "careful vs careless" transfers to "right
vs wrong" among the model's *own* plain-prompt answers.

We use the `trl` reward model for RL below. The hand-written one is no longer
needed, so we free its memory.

In [ ]:
del rm_torch, rm_opt
free_memory()

## 6. Step 4a — RL on plain prompts with the RLCD reward model (`GRPOTrainer`)

The RLCD paper used PPO (notebook 12.3 implements PPO). Here we reuse
notebook 12's GRPO setup, with one change: **the reward is the RLCD reward
model's score**, not the verifier. Two details are worth spelling out:

- **`scale_rewards=True`** (divide by the group's standard deviation). Reward
  model scores have an arbitrary scale (they're logits). Normalizing within each
  group makes the update size independent of that scale. With the verifier
  (notebook 12) the scale was fixed, so we didn't need it.
- **The verifier as a *monitor* with weight 0.** `reward_weights=[1.0, 0.0]`
  means the verifier's correctness is computed and **logged** at every step
  (`rewards/verifier_monitor/mean`), but contributes **nothing** to the
  training signal. You get to watch the true metric while the model optimizes
  the proxy — the healthy way to catch Goodhart effects early (notebook 7).

In [ ]:
RL_STEPS = 200          # same budget as notebook 12: 200 steps x 16 sampled answers

def rlcd_rm_reward(prompts, completions, **kwargs):
    """The ONLY training signal: the RLCD reward model's score for (plain prompt, completion)."""
    questions = [p[-1]["content"] for p in prompts]
    responses = [c[0]["content"] for c in completions]
    return rm_scores_generic(rm_trl_score, questions, responses).tolist()

def verifier_monitor(completions, answer, **kwargs):
    """Logged for monitoring only (weight 0): is the answer actually right?"""
    return [correctness_score(c[0]["content"], a) for c, a in zip(completions, answer)]

rl_dataset = Dataset.from_list([{"prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}],
                                 "answer": a} for q, a in TRAIN])
grpo_config = GRPOConfig(
    output_dir="/tmp/nb12_2_rlcd_grpo",
    num_generations=8, per_device_train_batch_size=16,     # 2 prompts x 8 completions per step (as notebook 12)
    max_completion_length=160, temperature=1.0,
    loss_type="dr_grpo",
    scale_rewards=True,                                      # RM scores have an arbitrary scale -> normalize per group
    reward_weights=[1.0, 0.0],                               # train on the RM; only MONITOR the verifier
    beta=0.04, epsilon=0.2, learning_rate=1e-5,
    max_steps=RL_STEPS, logging_steps=1,
    report_to="none", save_strategy="no", seed=0,
    bf16=USE_BF16, use_cpu=(DEVICE == "cpu"),
)
t0 = time.time()
grpo_trainer = GRPOTrainer(model=AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE),
                           reward_funcs=[rlcd_rm_reward, verifier_monitor], args=grpo_config,
                           train_dataset=rl_dataset, processing_class=tokenizer, peft_config=lora_cfg())
grpo_trainer.train()
rlcd_grpo_minutes = (time.time() - t0) / 60
print(f"\nRLCD + GRPO trained in {rlcd_grpo_minutes:.1f} min")

grpo_log = [r for r in grpo_trainer.state.log_history if "rewards/rlcd_rm_reward/mean" in r]
keys = {"rewards/rlcd_rm_reward/mean": "RM score (trained on)", "rewards/verifier_monitor/mean": "TRUE correctness",
        "completions/mean_length": "length", "kl": "KL"}
print(f"\n{'step':>4} | " + " | ".join(f"{v:>22}" for v in keys.values()))
for r in grpo_log[::10] + grpo_log[-1:]:
    print(f"{int(r['step']):>4} | " + " | ".join(f"{r.get(k, float('nan')):22.3f}" for k in keys))

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, (k, title) in zip(axes, [("rewards/rlcd_rm_reward/mean", "RLCD reward-model score (the proxy we optimize)"),
                                 ("rewards/verifier_monitor/mean", "TRUE correctness (monitor only, weight 0)"),
                                 ("completions/mean_length", "completion length (tokens)")]):
    ys = np.array([r.get(k, np.nan) for r in grpo_log], dtype=float)
    ax.plot(ys, alpha=0.3); ax.plot(np.convolve(ys, np.ones(5) / 5, mode="valid"))
    ax.set_title(title, fontsize=10); ax.set_xlabel("step")
plt.tight_layout(); plt.show()

rlcd_grpo_policy = grpo_trainer.model.eval()
rlcd_grpo_metrics, rlcd_grpo_responses = evaluate(rlcd_grpo_policy, "AFTER RLCD + GRPO (trl)")

**How to read this run.** The left plot always goes up — it's what we
optimize. The middle plot is the honest one. If true correctness rises with the
RM score, the RLCD reward captured something real about reasoning. If the RM
score rises while correctness stalls or falls, the policy is exploiting the
reward model (for example, by writing long, careful-*looking* reasoning that's
wrong). That's notebook 7's Goodhart curve again. Watch the length plot
alongside it.

## 7. Step 4b — Skip the reward model: DPO directly on the RLCD pairs

DPO (notebook 10) turns the same pairs into a supervised loss on the policy
itself. There's no reward model and no sampling during training:

$$
\mathcal{L}_{\text{DPO}}(\theta) = - \mathbb{E} \left[ \log \sigma \left( \beta \left[ \log \frac{\pi_\theta(y_+ | x)}{\pi_{\text{ref}}(y_+ | x)} - \log \frac{\pi_\theta(y_- | x)}{\pi_{\text{ref}}(y_- | x)} \right] \right) \right], \qquad \beta = 0.1 .
$$

### 7a. With `trl.DPOTrainer`

In [ ]:
DPO_EPOCHS = 1
dpo_args = DPOConfig(
    output_dir="/tmp/nb12_2_dpo_trl", beta=0.1,
    num_train_epochs=DPO_EPOCHS, per_device_train_batch_size=8, learning_rate=2e-5,
    max_length=512, logging_steps=5, report_to="none", save_strategy="no", seed=0,
    bf16=USE_BF16, use_cpu=(DEVICE == "cpu"),
)
t0 = time.time()
dpo_trainer = DPOTrainer(model=AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE), args=dpo_args,
                         train_dataset=pairs_ds, processing_class=tokenizer, peft_config=lora_cfg())
dpo_trainer.train()
dpo_trl_minutes = (time.time() - t0) / 60
print(f"\nDPO (trl) trained in {dpo_trl_minutes:.1f} min")
for row in dpo_trainer.state.log_history:
    if "loss" in row:
        print({k: round(float(row[k]), 4) for k in ["step", "loss", "rewards/accuracies", "rewards/margins",
                                                     "logps/chosen", "logps/rejected"] if k in row})

dpo_trl_policy = dpo_trainer.model.eval()
dpo_trl_metrics, dpo_trl_responses = evaluate(dpo_trl_policy, "AFTER RLCD + DPO (trl)")

### 7b. DPO by hand: `nn.Module` + training loop

The only real work is computing $\log \pi(y \mid x)$ for a whole response: the
**sum of the log-probabilities of its tokens**, given the prompt.

1. Tokenize the prompt and the response separately, concatenate, and remember
   which positions belong to the response (the *completion mask*).
2. One forward pass gives logits at every position. The logit at position
   $t-1$ predicts token $t$ (shift by one).
3. Selective log-softmax (notebook 12) for the true next tokens, multiplied by
   the completion mask, summed.

The reference log-probs are computed **once**, before training, with the LoRA
adapters disabled. They're constants of the loss.

In [ ]:
class LoRAPolicy(nn.Module):
    """A LoRA-adapted causal LM that returns sequence log-probs log pi(y|x) of given responses."""

    def __init__(self, model_name):
        super().__init__()
        self.lm = get_peft_model(AutoModelForCausalLM.from_pretrained(model_name, dtype=DTYPE), lora_cfg())
        lora_params_to_fp32(self.lm)

    def forward(self, input_ids, attention_mask, completion_mask):
        logits = self.lm(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1]
        token_logp = selective_log_softmax(logits, input_ids[:, 1:])            # position t-1 predicts token t
        return (token_logp * completion_mask[:, 1:]).sum(dim=1)                  # sum over response tokens only

    @torch.no_grad()
    def reference(self, *args):
        with self.lm.disable_adapter():                                          # reference = the base model
            return self.forward(*args)

def encode_pairs(rows, key):
    """Right-padded batch of prompt+response, with a mask marking the response tokens."""
    seqs, cmasks = [], []
    for r in rows:
        p_ids = tokenizer.apply_chat_template(r["prompt"], tokenize=False, add_generation_prompt=True)
        p_ids = tokenizer(p_ids, add_special_tokens=False)["input_ids"]
        c_ids = tokenizer(r[key][0]["content"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"][:200]
        seqs.append(p_ids + c_ids); cmasks.append([0] * len(p_ids) + [1] * len(c_ids))
    T = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), T), tokenizer.pad_token_id); attn = torch.zeros(len(seqs), T, dtype=torch.long)
    cm = torch.zeros(len(seqs), T)
    for i, (s, c) in enumerate(zip(seqs, cmasks)):
        ids[i, :len(s)] = torch.tensor(s); attn[i, :len(s)] = 1; cm[i, :len(c)] = torch.tensor(c, dtype=torch.float)
    return ids.to(DEVICE), attn.to(DEVICE), cm.to(DEVICE)

set_seed(0)
dpo_policy = LoRAPolicy(MODEL_NAME).to(DEVICE)
dpo_opt = torch.optim.AdamW([p for p in dpo_policy.parameters() if p.requires_grad], lr=2e-5)
BETA, BATCH, MICRO = 0.1, 8, 4

# Reference log-probs: computed once, with adapters disabled.
t0 = time.time()
ref_chosen, ref_rejected = [], []
for i in range(0, len(train_pairs), MICRO):
    rows = train_pairs[i:i + MICRO]
    ref_chosen.append(dpo_policy.reference(*encode_pairs(rows, "chosen")))
    ref_rejected.append(dpo_policy.reference(*encode_pairs(rows, "rejected")))
ref_chosen, ref_rejected = torch.cat(ref_chosen), torch.cat(ref_rejected)
print(f"reference log-probs for {len(train_pairs)} pairs in {time.time() - t0:.0f}s "
      f"(mean log pi_ref: chosen {ref_chosen.mean():.1f}, rejected {ref_rejected.mean():.1f})")

step = 0
for epoch in range(DPO_EPOCHS):
    order = list(range(len(train_pairs))); random.Random(epoch).shuffle(order)
    for i in range(0, len(order), BATCH):
        idx = order[i:i + BATCH]
        dpo_policy.train(); dpo_opt.zero_grad()
        stats = defaultdict(list)
        for j in range(0, len(idx), MICRO):                                     # gradient accumulation
            sub = idx[j:j + MICRO]; rows = [train_pairs[k] for k in sub]
            lp_c = dpo_policy(*encode_pairs(rows, "chosen")); lp_r = dpo_policy(*encode_pairs(rows, "rejected"))
            z = BETA * ((lp_c - ref_chosen[sub]) - (lp_r - ref_rejected[sub]))  # implicit reward margin
            loss = -F.logsigmoid(z).mean()
            (loss * len(sub) / len(idx)).backward()
            stats["loss"].append(loss.item()); stats["acc"] += (z > 0).float().tolist()
            stats["d_chosen"] += (lp_c - ref_chosen[sub]).tolist(); stats["d_rejected"] += (lp_r - ref_rejected[sub]).tolist()
        torch.nn.utils.clip_grad_norm_(dpo_policy.parameters(), 1.0); dpo_opt.step()
        if step % 5 == 0:
            print(f"step {step:3d} | loss {np.mean(stats['loss']):.4f} | implicit-reward acc {np.mean(stats['acc']):.2f} | "
                  f"d log p: chosen {np.mean(stats['d_chosen']):+.2f}, rejected {np.mean(stats['d_rejected']):+.2f}")
        step += 1
dpo_torch_minutes = (time.time() - t0) / 60
print(f"\nDPO (PyTorch) trained in {dpo_torch_minutes:.1f} min")
dpo_torch_metrics, dpo_torch_responses = evaluate(dpo_policy.lm, "AFTER RLCD + DPO (PyTorch)")

Watch the last two columns of the DPO log. DPO only needs the *margin* to grow.
If the log-probability of the **chosen** responses also *falls* (just less than
the rejected ones), that's the **likelihood displacement** from notebook 10:
probability mass moves to responses that appear in neither set.

## 8. Before vs. after — and saving results for notebook 12.3

In [ ]:
rows = {"base model (before)": before_metrics, "RLCD + GRPO (trl)": rlcd_grpo_metrics,
        "RLCD + DPO (trl)": dpo_trl_metrics, "RLCD + DPO (PyTorch)": dpo_torch_metrics}
cols = list(before_metrics.keys())
print(f"{'':24} | " + " | ".join(f"{c:>31}" for c in cols))
for name, m in rows.items():
    print(f"{name:24} | " + " | ".join(f"{m[c]:31.3f}" for c in cols))

for idx in [0, 1]:
    q, a = TEST[idx]
    print("\n" + "=" * 100 + f"\nQUESTION: {q}\nTRUE ANSWER: {a}")
    for name, resp in [("BEFORE", before_responses), ("RLCD + GRPO", rlcd_grpo_responses),
                       ("RLCD + DPO (trl)", dpo_trl_responses), ("RLCD + DPO (PyTorch)", dpo_torch_responses)]:
        print(f"\n--- {name} [{'correct' if parse_answer(resp[idx]) == a else 'not correct'}]\n{resp[idx].strip()[:500]}")

In [ ]:
n_pairs = len(train_pairs)
methods = {
    "RLCD + GRPO (trl)": method_record(
        rlcd_grpo_metrics, rlcd_grpo_responses, algorithm="GRPO (reward = RLCD reward model)", implementation="trl",
        reward_signal="RLCD reward model (no verifier)", train_minutes=rm_trl_minutes + rlcd_grpo_minutes,
        trainable_params=count_trainable(rlcd_grpo_policy), samples_generated=2 * n_pairs + RL_STEPS * 16,
        models_in_memory="policy (LoRA) + reward model (LoRA); reference = adapters off",
        notes=f"RM held-out pair acc {rm_trl_probe['heldout_pair_acc']:.2f}, correctness AUC {rm_trl_probe['correctness_auc']:.2f}"),
    "RLCD + DPO (trl)": method_record(
        dpo_trl_metrics, dpo_trl_responses, algorithm="DPO on RLCD pairs", implementation="trl",
        reward_signal="RLCD pairs (no verifier, no reward model)", train_minutes=dpo_trl_minutes,
        trainable_params=count_trainable(dpo_trl_policy), samples_generated=2 * n_pairs,
        models_in_memory="policy (LoRA); reference = adapters off"),
    "RLCD + DPO (PyTorch)": method_record(
        dpo_torch_metrics, dpo_torch_responses, algorithm="DPO on RLCD pairs", implementation="PyTorch",
        reward_signal="RLCD pairs (no verifier, no reward model)", train_minutes=dpo_torch_minutes,
        trainable_params=count_trainable(dpo_policy), samples_generated=2 * n_pairs,
        models_in_memory="policy (LoRA); reference = adapters off"),
}
save_results("12_2_rlcd.json", methods)

rlcd_grpo_policy.save_pretrained(CKPT_DIR / "12_2_rlcd_grpo_trl")
dpo_trl_policy.save_pretrained(CKPT_DIR / "12_2_rlcd_dpo_trl")
dpo_policy.lm.save_pretrained(CKPT_DIR / "12_2_rlcd_dpo_torch")
rm_model.save_pretrained(CKPT_DIR / "12_2_rlcd_reward_model")
print("saved LoRA adapters to", CKPT_DIR.resolve())

## 9. Key takeaways & FAQ

**Q1. What was RLCD supposed to teach here, and how do I know whether it did?**
The contrast was "careful, structured, double-checked reasoning" versus "a quick
guess." Success means the plain-prompt model now reasons in the structured
format **and** gets more answers right, *without* ever being rewarded for
correctness. Compare the strict accuracy in Section 8 with the base model's,
and with notebook 12's GRPO, which *was* rewarded for correctness. Notebook
12.3 puts them side by side.

**Q2. Why can RLCD improve correctness at all, if no one checks the answers?**
Because careful reasoning *correlates* with correct answers (look at the
accuracy of $o_+$ vs $o_-$ in the diagnostic). The reward model learns the
*style* of careful work, and that style carries some of the correctness.
The **correctness AUC** measures how much: well above 0.5 means it transferred;
close to 0.5 means the reward model rewards careful-*looking* answers whether
or not they're right.

**Q3. What are the typical failure modes?**
- **A weak contrast** (the model ignores $p_-$) → the labels are noise (Section 3).
- **A spurious cue**, especially *length* → the policy learns to write more, not
  better. Check the RM-score/length correlation and the length plot.
- **Reward hacking** of the reward model → the RM score rises while the
  monitored true correctness doesn't (Section 6's middle plot).

**Q4. GRPO with the reward model, or DPO on the pairs?**
- DPO is cheaper: no reward model, no sampling during training.
- The RL route keeps generating from the *current* policy, so it can keep
  improving beyond the fixed pairs. It can also over-optimize the reward model;
  the KL penalty and the verifier monitor are your guards.
- DPO only sees the original pairs, so it can't overshoot the reward model.
  But it can show likelihood displacement (Section 7b's log).

**Q5. Why `scale_rewards=True` here but not in notebook 12?**
Reward-model scores are logits with an arbitrary offset and scale. Dividing
by the group's standard deviation makes each update independent of that
arbitrary scale. The verifier's rewards in notebook 12 had a fixed, meaningful
scale (0–1.75), so Dr. GRPO's unscaled version was preferable there.

**Q6. Which KL is used, and where?**
In the GRPO run, the same as notebook 12: the k3 estimate of the **reverse**
KL, $\mathrm{KL}(\pi_\theta \Vert \pi_{\text{ref}})$, added to the loss with $\beta = 0.04$.
In DPO there's no explicit KL term. The $\beta$ inside the loss plays that role
implicitly, because DPO is derived from the same KL-regularized objective
(notebook 10).